# Lezione 2 — Classificazione supervisionata end-to-end

**Obiettivo:** costruire il primo ciclo completo di Machine Learning su un dataset tabellare:
- caricamento e comprensione del dataset
- train/test split
- scaling
- addestramento di due modelli
- valutazione e confronto
- interpretazione finale

## Dataset
Useriamo il dataset **Wine** di `scikit-learn`, un classico problema di classificazione multiclasse.

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option("display.max_columns", 50)

## 1) Caricamento dati

In [15]:
wine = load_wine(as_frame=True)
df = wine.frame.copy()
df["class_name"] = df["target"].map(dict(enumerate(wine.target_names)))
wine.frame.head(), print(wine.target_names) ,df.head()


['class_0' 'class_1' 'class_2']


(   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
 0    14.23        1.71  2.43               15.6      127.0           2.80   
 1    13.20        1.78  2.14               11.2      100.0           2.65   
 2    13.16        2.36  2.67               18.6      101.0           2.80   
 3    14.37        1.95  2.50               16.8      113.0           3.85   
 4    13.24        2.59  2.87               21.0      118.0           2.80   
 
    flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
 0        3.06                  0.28             2.29             5.64  1.04   
 1        2.76                  0.26             1.28             4.38  1.05   
 2        3.24                  0.30             2.81             5.68  1.03   
 3        3.49                  0.24             2.18             7.80  0.86   
 4        2.69                  0.39             1.82             4.32  1.04   
 
    od280/od315_of_diluted_wines  proline  targe

In [12]:
df.shape, df["class_name"].value_counts()

((178, 15),
 class_name
 class_1    71
 class_0    59
 class_2    48
 Name: count, dtype: int64)

### Domande
1. Quante classi ci sono?
2. Il dataset è bilanciato?
3. Qual è il target?

In [ ]:
#Guardiamo come sono le feature
df.describe().loc[["min", "max"]].T.assign(range=lambda x: x["max"] - x["min"]).sort_values("range", ascending=False)

,min,max,range
proline,278.00,1680.00,1402.00
magnesium,70.00,162.00,92.00
alcalinity_of_ash,10.60,30.00,19.40
color_intensity,1.28,13.00,11.72
malic_acid,0.74,5.80,5.06
flavanoids,0.34,5.08,4.74
alcohol,11.03,14.83,3.80
proanthocyanins,0.41,3.58,3.17
total_phenols,0.98,3.88,2.90
od280/od315_of_diluted_wines,1.27,4.00,2.73


![alt text](image.png)

## 2) Definizione di X e y

In [22]:
X = df.drop(columns=["target", "class_name"])
y = df["class_name"]

X.head(), y.head(),X.shape, y.shape

(   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
 0    14.23        1.71  2.43               15.6      127.0           2.80   
 1    13.20        1.78  2.14               11.2      100.0           2.65   
 2    13.16        2.36  2.67               18.6      101.0           2.80   
 3    14.37        1.95  2.50               16.8      113.0           3.85   
 4    13.24        2.59  2.87               21.0      118.0           2.80   
 
    flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
 0        3.06                  0.28             2.29             5.64  1.04   
 1        2.76                  0.26             1.28             4.38  1.05   
 2        3.24                  0.30             2.81             5.68  1.03   
 3        3.49                  0.24             2.18             7.80  0.86   
 4        2.69                  0.39             1.82             4.32  1.04   
 
    od280/od315_of_diluted_wines  proline  
 0  

## 3) Train/Test split

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((133, 13), (45, 13))

**Nota:** usiamo `stratify=y` per mantenere proporzioni simili delle classi in train e test.

## 4) Scaling

In [54]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

X_train_s[:3]

array([[ 2.33041778, -0.63796267, -0.74427132, -1.70005902, -0.20099315,
         0.82455876,  1.033603  , -0.60488012,  0.64316856,  0.08154467,
         0.57887517,  0.37385664,  0.97971254],
       [-0.57363985, -0.55510229, -1.45939437,  0.23809346, -1.00496573,
        -0.13141667, -0.07687826, -0.35083047, -0.21344488, -0.87740273,
         0.4011093 ,  1.42836362, -0.248245  ],
       [ 0.39020686, -0.63796267,  1.77747839, -1.25279306,  0.66997716,
         0.50590028,  0.71931585, -0.18146404, -0.41903211, -0.17304314,
         0.62331663,  0.27133513,  0.43651416]])

## 5) Modello 1 — Logistic Regression

In [55]:
logreg = LogisticRegression(max_iter=3500)
logreg.fit(X_train_s, y_train)

pred_log = logreg.predict(X_test_s)
acc_log = accuracy_score(y_test, pred_log)
acc_log

1.0

In [56]:
print(classification_report(y_test, pred_log))

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        15
     class_1       1.00      1.00      1.00        18
     class_2       1.00      1.00      1.00        12

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



## 6) Modello 2 — kNN

In [76]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)

pred_knn = knn.predict(X_test_s)
acc_knn = accuracy_score(y_test, pred_knn)
acc_knn

0.9333333333333333

In [77]:
print(classification_report(y_test, pred_knn))

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        15
     class_1       0.94      0.89      0.91        18
     class_2       0.85      0.92      0.88        12

    accuracy                           0.93        45
   macro avg       0.93      0.94      0.93        45
weighted avg       0.94      0.93      0.93        45



## 7) Confronto modelli

In [78]:
results = pd.DataFrame({
    "model": ["Logistic Regression", "kNN (k=5)"],
    "accuracy": [acc_log, acc_knn]
}).sort_values("accuracy", ascending=False)

results

,model,accuracy
0,Logistic Regression,1.000000
1,kNN (k=5),0.933333


In [83]:
#Confusion Matrix LogReg

best_model = logreg if acc_log >= acc_knn else knn
best_pred = pred_log if acc_log >= acc_knn else pred_knn
best_name = "Logistic Regression" if acc_log >= acc_knn else "kNN (k=5)"

cm = confusion_matrix(y_test, best_pred, labels=best_model.classes_)
cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in best_model.classes_], columns=[f"pred_{c}" for c in best_model.classes_])
best_name, cm_df

('Logistic Regression',
               pred_class_0  pred_class_1  pred_class_2
 true_class_0            15             0             0
 true_class_1             0            18             0
 true_class_2             0             0            12)

In [81]:
#Confusion Matrix KNN

worst_model = logreg if acc_log < acc_knn else knn
worst_pred = pred_log if acc_log < acc_knn else pred_knn
worst_name = "Logistic Regression" if acc_log < acc_knn else "kNN (k=5)"

cm = confusion_matrix(y_test, worst_pred, labels=worst_model.classes_)
cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in worst_model.classes_], columns=[f"pred_{c}" for c in worst_model.classes_])
worst_name, cm_df

('kNN (k=5)',
               pred_class_0  pred_class_1  pred_class_2
 true_class_0            15             0             0
 true_class_1             0            16             2
 true_class_2             0             1            11)

## 8) Interpretazione

In [88]:
if best_name == "Logistic Regression":
    coef_df = pd.DataFrame(logreg.coef_, columns=X.columns, index=logreg.classes_)
    print(coef_df.iloc[:, :13])
else:
    print("Per kNN non abbiamo coefficienti: l'interpretazione passa soprattutto da errori e confronto metriche.")



          alcohol  malic_acid       ash  alcalinity_of_ash  magnesium  \
class_0  0.721026    0.163307  0.467236          -0.813141   0.080950   
class_1 -0.841909   -0.509378 -0.789258           0.514494  -0.123389   
class_2  0.120883    0.346071  0.322021           0.298647   0.042438   

         total_phenols  flavanoids  nonflavanoid_phenols  proanthocyanins  \
class_0       0.220783    0.701163             -0.105280         0.138645   
class_1       0.051844    0.187726              0.127115         0.374500   
class_2      -0.272628   -0.888889             -0.021835        -0.513145   

         color_intensity       hue  od280/od315_of_diluted_wines   proline  
class_0         0.194979  0.087458                      0.651255  0.918558  
class_1        -1.020914  0.677793                     -0.111763 -0.937545  
class_2         0.825935 -0.765251                     -0.539492  0.018986  


## 9) Comunicazione dei risultati

Scrivi una conclusione breve:
1. Quale modello scegli?
2. Perché?
3. Dove sbaglia?
4. Quali caratteristiche del dataset potrebbero spiegare questi errori?